In [5]:
import pandas as pd
from pathlib import Path
from typing import List, Set

In [6]:
BASE_DIR   = Path("/cluster2/home/zeyu/Projects/Program/cross_species_integration/Matching/Methods/HM_O2O")
O2O_DIR    = BASE_DIR / "cross_species_homologue_genes"
RESULT_DIR = O2O_DIR / "o2oResults"         
RESULT_DIR.mkdir(exist_ok=True)
PAIRS = [
    # ("human",               "mice"),
    # ("chimpanzee",          "gorilla"),
    # ("crab_eating_macaque", "mice"),
    # ("crab_eating_macaque", "rhesus_macaque"),
    # ("gorilla",             "rhesus_macaque"),
    # ("human",               "chimpanzee"),
    # ("human",               "microcebus"),
    # ("microcebus",          "human"),
    # ("mice",                "frog"),
    # ("mice",                "pig"),
    # ("rhesus_macaque",      "human"),
    # ("rhesus_macaque",      "commonmarmoset"),
    #("human",               "pig"),
    ("zebrafish",          "frog")
]

In [7]:
def greedy_with_confidence(
    df: pd.DataFrame,
    first_col: str,
    second_col: str,
    primary_score_col: str = "orthology confidence",
    identical_col: str = "identical_scores",
) -> pd.DataFrame:
    """高置信优先 + 全局 greedy → 严格 1-to-1 配对"""
    df_sorted = (
        df.sort_values(
            by=[primary_score_col, identical_col],
            ascending=[False, False],
            kind="mergesort",
        )
        .drop_duplicates(subset=[first_col, second_col], keep="first")
        .reset_index(drop=True)
    )

    high_conf = df_sorted[df_sorted[primary_score_col] == 1]
    low_conf  = df_sorted[df_sorted[primary_score_col] == 0]

    used_f: Set[str] = set()
    used_s: Set[str] = set()
    chosen: List[pd.Series] = []

    # —— 第 1 阶段：置信度 1 —— #
    for _, row in high_conf.sort_values(identical_col, ascending=False).iterrows():
        f, s = row[first_col], row[second_col]
        if f not in used_f and s not in used_s:
            chosen.append(row)
            used_f.add(f); used_s.add(s)

    # —— 第 2 阶段：置信度 0 —— #
    remaining = (
        low_conf[~low_conf[first_col].isin(used_f) & ~low_conf[second_col].isin(used_s)]
        .sort_values(identical_col, ascending=False)
    )
    for _, row in remaining.iterrows():
        f, s = row[first_col], row[second_col]
        if f not in used_f and s not in used_s:
            chosen.append(row)
            used_f.add(f); used_s.add(s)

    return pd.DataFrame(chosen)[[first_col, second_col, identical_col]]

In [8]:
for sp1, sp2 in PAIRS:
    print(f"\n=== {sp1} ↔ {sp2} ===")
    src_file = O2O_DIR / f"{sp1}_{sp2}_ENS_M2M.txt"
    if not src_file.exists():
        print(f"  ❌ 源文件缺失：{src_file.name}，跳过")
        continue

    raw = pd.read_csv(src_file, sep="\t")
    raw = raw.dropna(subset=raw.columns[:2])
    raw.rename(columns={raw.columns[-1]: "orthology confidence"}, inplace=True)
    # 第 4、5 列转数值后求平均
    raw[[raw.columns[3], raw.columns[4]]] = raw[[raw.columns[3], raw.columns[4]]].apply(
        pd.to_numeric, errors="coerce"
    )
    raw["identical_scores"] = raw[[raw.columns[3], raw.columns[4]]].mean(axis=1)
    # orthology confidence → int(0/1)
    raw["orthology confidence"] = (
        pd.to_numeric(raw["orthology confidence"], errors="coerce").fillna(0).astype(int)
    )

    # ---------- 2.3 全局 greedy ----------
    matched = greedy_with_confidence(
        raw,
        first_col=raw.columns[0],
        second_col=raw.columns[1],
    )

    # ---------- 2.4 保存新结果 ----------
    matched.rename(
        columns={
            matched.columns[0]: f"{sp1} gene name",
            matched.columns[1]: f"{sp2} gene name",
        },
        inplace=True,
    )
    matched = matched.iloc[:, :2]        # 只保留基因名两列
    new_path = RESULT_DIR / f"{sp1}_{sp2}_o2o.csv"
    matched.to_csv(new_path, index=False)
    print(f"  ✅ 新结果保存：{new_path.name}（{matched.shape[0]} 对）")

    # ---------- 2.5 与旧版结果比对 ----------
    old_path = RESULT_DIR / f"{sp1}_{sp2}.csv"
    if not old_path.exists():
        print("  ⚠ 未找到旧版结果，跳过一致性检查")
        continue

    old = pd.read_csv(old_path).iloc[:, :2]
    old.columns = [f"{sp1} gene name", f"{sp2} gene name"]

    set_new = set(matched.apply(tuple, axis=1))
    set_old = set(old.apply(tuple, axis=1))

    diff_new = set_new - set_old          # 新增配对
    diff_old = set_old - set_new          # 旧有但现在消失的配对

    incons_rows = (
        [(a, b, "o2o_new")     for a, b in diff_new] +
        [(a, b, "o2o_before")  for a, b in diff_old]
    )
    if incons_rows:
        incons_df = pd.DataFrame(
            incons_rows, columns=[f"{sp1} gene name", f"{sp2} gene name", "source"]
        )
        inc_path = RESULT_DIR / f"{sp1}_{sp2}_inconsistent.txt"
        incons_df.to_csv(inc_path, sep="\t", index=False)
        print(f"  ⚠ 不一致配对 {incons_df.shape[0]} 条 → {inc_path.name}")
    else:
        print("  👍 新旧结果完全一致")
    # ---------- 2.6 检查是否有重复基因 ----------
    col_f, col_s = matched.columns[:2]        # 第一列、第二列列名

    dup_f = matched[col_f].duplicated(keep=False)
    dup_s = matched[col_s].duplicated(keep=False)

    if dup_f.any() or dup_s.any():
        dup_df = matched[dup_f | dup_s]
        dup_path = RESULT_DIR / f"{sp1}_{sp2}_duplicates.txt"
        dup_df.to_csv(dup_path, sep="\t", index=False)
        print(f"  ⚠ 发现重复基因！已保存 {dup_path.name}（{dup_df.shape[0]} 行）")
    else:
        print("  ✅ 未发现重复基因")


=== zebrafish ↔ frog ===
  ✅ 新结果保存：zebrafish_frog_o2o.csv（12517 对）
  ⚠ 未找到旧版结果，跳过一致性检查
